In [2]:
import os
try:
    path_initialized
except NameError:
    path_initialized = True
    os.chdir('..')

import numpy as np
import sinter
import stim
import os
from datetime import date
from typing import List, Union

import numpy as np
from sympy.abc import x, y

from qldpc import codes
from qldpc.objects import Pauli
import networkx as nx
import matplotlib.pyplot as plt

import src.device as device
import src.plotting as plotter
from src.RotatedSurfaceCode import RotatedSurfaceCode
from src.HGPCode import HGPCode
from src.GBCode import GBCode
from src.QECCode import TestCode

import sys
from pathlib import Path
external_path = Path("src/extern/radial/decoding").resolve()
if str(external_path) not in sys.path:
    sys.path.insert(0, str(external_path))
external_path = Path("src/extern/radial/generation").resolve()
if str(external_path) not in sys.path:
    sys.path.insert(0, str(external_path))

from src.extern.radial.decoding.ckt_noise import SinterDecoder_BPOSD_OWD
import src.extern.radial.decoding.circuit_stuff as cs
import src.extern.radial.generation.codegen as codegen

In [ ]:
import numpy as np
import csv
from pathlib import Path

def save_matrix_as_csv(matrix, filepath):
    """Save a binary matrix as CSV with 0,1 format."""
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        for row in matrix:
            writer.writerow(row.astype(int))

def rref_gf2(matrix):
    """Compute the Reduced Row Echelon Form of a matrix over GF(2)."""
    A = matrix.copy() % 2
    rows, cols = A.shape
    r = 0
    pivots = []
    for c in range(cols):
        if r >= rows: break
        
        pivot_row = r
        while pivot_row < rows and A[pivot_row, c] == 0:
            pivot_row += 1
            
        if pivot_row == rows: continue
        
        # Swap current row with pivot row
        A[[r, pivot_row]] = A[[pivot_row, r]]
        pivots.append(c)
        
        # Eliminate other 1s in the current column
        for i in range(rows):
            if i != r and A[i, c] == 1:
                A[i] = (A[i] + A[r]) % 2
        r += 1
        
    return A, pivots

def nullspace_gf2(matrix):
    """Compute the null space of a binary matrix over GF(2)."""
    A, pivots = rref_gf2(matrix)
    rows, cols = A.shape
    rank = len(pivots)
    nullity = cols - rank
    
    if nullity == 0:
        return np.zeros((0, cols), dtype=int)
    
    free_vars = [c for c in range(cols) if c not in pivots]
    ns = np.zeros((nullity, cols), dtype=int)
    
    for i, fv in enumerate(free_vars):
        ns[i, fv] = 1
        for j, pv in enumerate(pivots):
            ns[i, pv] = A[j, fv]
            
    return ns

def compute_logicals(hx, hz):
    """
    Compute CSS logical operators Lx and Lz.
    Lx in ker(Hz) and Lz in ker(Hx) such that Lx @ Lz.T = I_k mod 2.
    """
    # 1. Lx candidates commute with Hz; Lz candidates commute with Hx
    Kx = nullspace_gf2(hz)
    Kz = nullspace_gf2(hx)
    
    # 2. Compute the symplectic intersection matrix M
    M = (Kx @ Kz.T) % 2
    
    # 3. Diagonalize M over GF(2) using row/col operations to find canonical pairs
    rows, cols = M.shape
    r = 0
    for _ in range(min(rows, cols)):
        pivot_found = False
        # Find a 1 in the remaining M[r:, r:] submatrix
        for i in range(r, rows):
            for j in range(r, cols):
                if M[i, j] == 1:
                    # Swap rows r and i (updates M and Kx)
                    if i != r:
                        M[[r, i]] = M[[i, r]]
                        Kx[[r, i]] = Kx[[i, r]]
                    # Swap cols r and j (updates M and Kz)
                    if j != r:
                        M[:, [r, j]] = M[:, [j, r]]
                        Kz[[r, j]] = Kz[[j, r]]
                    pivot_found = True
                    break
            if pivot_found:
                break
        
        if not pivot_found:
            break # Reached the rank of M (this rank is exactly 'k')
            
        # Eliminate all other 1s in column r
        for i in range(rows):
            if i != r and M[i, r] == 1:
                M[i] = (M[i] + M[r]) % 2
                Kx[i] = (Kx[i] + Kx[r]) % 2
                
        # Eliminate all other 1s in row r
        for j in range(cols):
            if j != r and M[r, j] == 1:
                M[:, j] = (M[:, j] + M[:, r]) % 2
                Kz[j] = (Kz[j] + Kz[r]) % 2
                
        r += 1
        
    # The first r rows now form our conjugate logical operator pairs
    return Kx[:r], Kz[:r], r

In [ ]:
import subprocess
import shutil
from pathlib import Path
import re
import csv

r = 3
s = 11
abdiff = True
checkdiff = True
n_codes = 50
matpath = 'notebooks/data/TMP_matrices/'

# Ensure matpath exists and convert to absolute path
matpath_abs = Path(matpath).resolve()
matpath_abs.mkdir(parents=True, exist_ok=True)

best_code_idx = None
best_distance = 0
code_distances = {}
best_matrices = {}  # Store HX, HZ for the best code

for i in range(n_codes):
    A = codegen.matgen(r,s,checkdiff)
    if (abdiff):
        B = codegen.matgen(r,s,checkdiff)
    else:
        B = A

    A_proto = np.array([[codegen.Circulant(s,x) for x in row] for row in A])
    B_proto = np.array([[codegen.Circulant(s,x) for x in row] for row in B])
    
    HX, HZ = codegen.lifted_product(A_proto,B_proto,s)
    HX = codegen.protograph_to_binary_matrix(HX,s)
    HZ = codegen.protograph_to_binary_matrix(HZ,s)

    codegen.write_PCM_as_mtx(r,s,A,B,HX,'x',i+1,path=matpath)
    codegen.write_PCM_as_mtx(r,s,A,B,HZ,'z',i+1,path=matpath)

    # Files are written as hx{i}.mtx and hz{i}.mtx
    hx_path = matpath_abs / f'hx{i+1}.mtx'
    hz_path = matpath_abs / f'hz{i+1}.mtx'
    
    if not hx_path.exists() or not hz_path.exists():
        print(f"Code {i+1}: Missing matrix files. Looking for {hx_path.name} and {hz_path.name}")
        print(f"Available files: {[f.name for f in matpath_abs.glob('*.mtx')]}")
        continue
    
    gap_script = f"""
LoadPackage("QDistRnd");;
lsX:=ReadMTXE("{str(hx_path)}");;
lsZ:=ReadMTXE("{str(hz_path)}");;
d1:=DistRandCSS(lsX[3],lsZ[3],1000,0,8);;
d2:=DistRandCSS(lsZ[3],lsX[3],1000,0,8);;
Print("X-distance: ", d1, "\\n");
Print("Z-distance: ", d2, "\\n");
quit;
"""
    
    try:
        # Disable GAP colors with environment variables
        env = os.environ.copy()
        env['CLICOLOR'] = '0'
        env['CLICOLOR_FORCE'] = '0'
        
        result = subprocess.run(['gap', '-q'], input=gap_script, capture_output=True, text=True, timeout=300, env=env)
        output = result.stdout
        error = result.stderr
        
        # Strip ANSI codes
        output = strip_ansi(output)
        
        # Debug: print output and error if output is empty
        if not output.strip():
            print(f"Code {i+1}: Empty stdout. stderr: {error[:200]}")
            continue
        
        # Parse distances from output
        x_dist = None
        z_dist = None
        for line in output.split('\n'):
            if 'X-distance:' in line:
                try:
                    x_dist = int(line.split()[-1])
                except ValueError:
                    pass
            elif 'Z-distance:' in line:
                try:
                    z_dist = int(line.split()[-1])
                except ValueError:
                    pass
        
        if x_dist and z_dist:
            min_dist = min(x_dist, z_dist)
            code_distances[i+1] = {'x': x_dist, 'z': z_dist, 'min': min_dist}
            print(f"Code {i+1}: X-distance={x_dist}, Z-distance={z_dist}, min={min_dist}")
            
            if min_dist > best_distance:
                best_distance = min_dist
                best_code_idx = i+1
                best_matrices = {'HX': HX, 'HZ': HZ, 'A': A, 'B': B}
        else:
            print(f"Code {i+1}: Failed to parse distances")
            print(f"Output was: {repr(output)}")

        if best_distance == 2*s:
            print('Saturated distance bound!')
            break
    except subprocess.TimeoutExpired:
        print(f"Code {i+1}: GAP computation timed out")
    except FileNotFoundError:
        print("GAP not found. Skipping distance computation.")
        break

# Keep only the best code
if best_distance:
    print(f"\n=== Best Code: {best_code_idx} with distance {best_distance} ===")
    print(f"Details: X={code_distances[best_code_idx]['x']}, Z={code_distances[best_code_idx]['z']}")
    
    # Save best code as CSV
    best_dir = Path(f'notebooks/data/radial_mats/r{r}_s{s}_d{best_distance}')
    best_dir.mkdir(exist_ok=True)
    
    hx_csv = best_dir / 'hx.csv'
    hz_csv = best_dir / 'hz.csv'
    save_matrix_as_csv(best_matrices['HX'], hx_csv)
    save_matrix_as_csv(best_matrices['HZ'], hz_csv)
    print(f"Saved HX to {hx_csv}")
    print(f"Saved HZ to {hz_csv}")
    
    lx, lz, k = compute_logicals(best_matrices['HX'], best_matrices['HZ'])

    expected_k = 2 * (r - 1) ** 2
    Q_hx = best_matrices['HX'].shape[1]
    Q_hz = best_matrices['HZ'].shape[1]

    print(f"({r}, {s}, {d}): Expected {expected_k} operators, Found {k}, Q_hx={Q_hx}, Q_hz={Q_hz}")
    print(f"  LX shape: {lx.shape}, expected ({expected_k}, {Q_hx})")
    print(f"  LZ shape: {lz.shape}, expected ({expected_k}, {Q_hz})")
    
    # 1. Structural Assertions
    assert lx.shape == (expected_k, Q_hx), f"LX shape {lx.shape} != expected ({expected_k}, {Q_hx})"
    assert lz.shape == (expected_k, Q_hz), f"LZ shape {lz.shape} != expected ({expected_k}, {Q_hz})"
    
    # 2. Strict QEC Mathematical Assertions
    assert not np.any((lx @ best_matrices['HZ'].T) % 2), "Error: Lx does not commute with Hz checks!"
    assert not np.any((lz @ best_matrices['HX'].T) % 2), "Error: Lz does not commute with Hx checks!"
    assert np.array_equal((lx @ lz.T) % 2, np.eye(k, dtype=int)), "Error: Lx and Lz do not form valid conjugate Pauli pairs!"
    
    if len(lx) > 0:
        save_matrix_as_csv(lx, best_dir / 'lx.csv')
    if len(lz) > 0:
        save_matrix_as_csv(lz, best_dir / 'lz.csv')

In [ ]:
# # Fix previously saved logical operators
# radial_mats_dir = Path('notebooks/data/radial_mats')
# skip_codes = {(3, 5, 10), (4, 11, 20)}

# if radial_mats_dir.exists():
#     for code_dir in sorted(radial_mats_dir.glob('r*_s*_d*')):
#         # Parse r, s, d from directory name
#         parts = code_dir.name.split('_')
#         r = int(parts[0][1:])
#         s = int(parts[1][1:])
#         d = int(parts[2][1:])
        
#         if (r, s, d) in skip_codes:
#             print(f"Skipping ({r}, {s}, {d})")
#             continue
        
#         hx_path = code_dir / 'hx.csv'
#         hz_path = code_dir / 'hz.csv'
        
#         if not hx_path.exists() or not hz_path.exists():
#             print(f"Missing HX or HZ for {code_dir.name}")
#             continue
        
#         # Load matrices
#         hx = np.loadtxt(hx_path, dtype=int, delimiter=',')
#         hz = np.loadtxt(hz_path, dtype=int, delimiter=',')
        
#         # Compute logicals
#         lx, lz, k = compute_logicals(hx, hz)
        
#         # Verify shapes
#         expected_k = 2 * (r - 1) ** 2
#         Q_hx = hx.shape[1]
#         Q_hz = hz.shape[1]
        
#         print(f"({r}, {s}, {d}): Expected {expected_k} operators, Found {k}, Q_hx={Q_hx}, Q_hz={Q_hz}")
#         print(f"  LX shape: {lx.shape}, expected ({expected_k}, {Q_hx})")
#         print(f"  LZ shape: {lz.shape}, expected ({expected_k}, {Q_hz})")
        
#         # 1. Structural Assertions
#         assert lx.shape == (expected_k, Q_hx), f"LX shape {lx.shape} != expected ({expected_k}, {Q_hx})"
#         assert lz.shape == (expected_k, Q_hz), f"LZ shape {lz.shape} != expected ({expected_k}, {Q_hz})"
        
#         # 2. Strict QEC Mathematical Assertions
#         assert not np.any((lx @ hz.T) % 2), "Error: Lx does not commute with Hz checks!"
#         assert not np.any((lz @ hx.T) % 2), "Error: Lz does not commute with Hx checks!"
#         assert np.array_equal((lx @ lz.T) % 2, np.eye(k, dtype=int)), "Error: Lx and Lz do not form valid conjugate Pauli pairs!"
        
#         if len(lx) > 0:
#             save_matrix_as_csv(lx, code_dir / 'lx.csv')
#         if len(lz) > 0:
#             save_matrix_as_csv(lz, code_dir / 'lz.csv')
            
#         print(f"  Fixed {code_dir.name}\n")
# else:
#     print(f"{radial_mats_dir} does not exist")

In [ ]:
from src.RadialCode import RadialCode
code_rad = RadialCode(3,11,14)

buffer = 3
xmax,ymax = 0,0
data_coords = dict()
for i in code_rad.data_indices:
    x,y = code_rad.qubit_coords[i]
    data_coords[i] = (x+buffer, y+buffer)
    xmax = max(xmax, x)
    ymax = max(ymax, y)

dev = device.UnitCellDevice(xmax+2*buffer, ymax+2*buffer, device.default_hwp)

sched_rad = dev.compile_QEC_schedule(
    code_rad,
    data_coords,
    [],
    rounds=10,
    use_highways=True,
    refocus_shuttle_noise=False,
    optimize_ancilla_start=True,
    separate_X_Z=True
)